# WM Prediction 2026 – Version 4: Quoten-Integration

| Feature | Beschreibung |
|---|---|
| **Quoten-Ensemble** | Pinnacle-Wahrscheinlichkeiten + Poisson+Elo kombiniert |
| **Ensemble-Gewichtung** | Modell 40% + Quoten 60% (kalibrierbar via ODDS_WEIGHT) |
| **Tippspiel-Optimierung** | Wahrscheinlichstes Ergebnis je Spiel |
| **Kompletter Gruppenplan** | Alle 48 Gruppenspiele getippt |

> Quoten fließen **nur** in die Match-Vorhersage ein – Elo und Poisson-Training bleiben unberührt.

## 1 · Setup

In [1]:
import pandas as pd
import numpy as np
import json
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')
from collections import defaultdict, deque
import matplotlib.pyplot as plt
import time

ODDS_WEIGHT  = 0.60
MODEL_WEIGHT = 1.0 - ODDS_WEIGHT

print(f'Pandas {pd.__version__}')
print(f'Ensemble: Modell {MODEL_WEIGHT:.0%}  +  Quoten {ODDS_WEIGHT:.0%}')

Pandas 2.3.2
Ensemble: Modell 40%  +  Quoten 60%


## 2 · Daten laden

In [2]:
all_matches = pd.read_csv('results.csv')
all_matches['date'] = pd.to_datetime(all_matches['date'])
all_matches = all_matches.sort_values('date').reset_index(drop=True)

cutoff = pd.Timestamp.now() - pd.DateOffset(years=20)
model_matches = all_matches[all_matches['date'] >= cutoff].copy()

print(f'Gesamt-Datensatz  (Elo):     {len(all_matches):>6} Spiele')
print(f'Modell-Datensatz  (Poisson): {len(model_matches):>6} Spiele')

Gesamt-Datensatz  (Elo):      49378 Spiele
Modell-Datensatz  (Poisson):  19341 Spiele


## 3 · Quoten laden & normalisieren

In [3]:
with open('odds.json', 'rb') as f:
    raw = f.read()

for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin-1'):
    try:
        odds_raw = json.loads(raw.decode(enc))
        break
    except Exception:
        continue

print(f'Quoten geladen: {odds_raw["fetched_at"]}')
print(f'Spiele: {len(odds_raw["games"])}')

# Overround entfernen
odds_lookup = {}
for g in odds_raw['games']:
    p = g['prob']
    total = p['home_win'] + p['draw'] + p['away_win']
    odds_lookup[(g['home_team'], g['away_team'])] = {
        'home_win': p['home_win'] / total,
        'draw':     p['draw']     / total,
        'away_win': p['away_win'] / total,
    }

# Datum-Map
odds_date_map = {}
for g in odds_raw['games']:
    odds_date_map[(g['home_team'], g['away_team'])] = g['commence_time']
    odds_date_map[(g['away_team'], g['home_team'])] = g['commence_time']

ex = odds_lookup.get(('Mexico', 'South Africa'), {})
print('\nBeispiel Mexico vs South Africa:')
for k, v in ex.items():
    print(f'  {k}: {v:.1%}')

Quoten geladen: 2026-06-05T19:41:45.074475+00:00
Spiele: 72

Beispiel Mexico vs South Africa:
  home_win: 66.9%
  draw: 21.5%
  away_win: 11.6%


## 4 · Elo-Berechnung

In [4]:
def get_k_factor(tournament):
    t = str(tournament)
    if t == 'FIFA World Cup':            return 60
    if 'UEFA Euro' in t:                 return 50
    if 'Copa America' in t:              return 50
    if 'African Cup of Nations' in t:    return 50
    if 'AFC Asian Cup' in t:             return 50
    if 'OFC Nations Cup' in t:           return 50
    if 'Nations League' in t:            return 40
    if 'qualification' in t.lower():     return 30
    if 'Friendly' in t:                  return 15
    return 25

def mov_multiplier(goal_diff, elo_diff_winner):
    if goal_diff == 0: return 1.0
    return np.log(goal_diff + 1) * (2.2 / (elo_diff_winner * 0.001 + 2.2))

INITIAL_ELO = 1500
elo = {}

def get_elo(team):
    return elo.get(team, INITIAL_ELO)

elo_home_before, elo_away_before = [], []
current_year = None

for _, row in all_matches.iterrows():
    year = row['date'].year
    if current_year is None:
        current_year = year
    if year > current_year:
        for team in elo:
            elo[team] = INITIAL_ELO + (elo[team] - INITIAL_ELO) * 0.98
        current_year = year
    home, away = row['home_team'], row['away_team']
    eh, ea = get_elo(home), get_elo(away)
    elo_home_before.append(eh)
    elo_away_before.append(ea)
    expected_home = 1 / (1 + 10 ** ((ea - eh) / 400))
    hs, as_ = row['home_score'], row['away_score']
    goal_diff = abs(hs - as_)
    if hs > as_:
        actual_home = 1.0
        mov = mov_multiplier(goal_diff, max(eh - ea, 0))
    elif hs < as_:
        actual_home = 0.0
        mov = mov_multiplier(goal_diff, max(ea - eh, 0))
    else:
        actual_home = 0.5
        mov = 1.0
    k = get_k_factor(row['tournament'])
    elo[home] = eh + k * mov * (actual_home - expected_home)
    elo[away] = ea + k * mov * ((1.0 - actual_home) - (1.0 - expected_home))

all_matches['elo_home'] = elo_home_before
all_matches['elo_away'] = elo_away_before
all_matches['elo_diff'] = all_matches['elo_home'] - all_matches['elo_away']
print(f'Elo abgeschlossen – {len(elo)} Teams.')

Elo abgeschlossen – 336 Teams.


## 5 · Elo-Tabelle

In [5]:
elo_table = (
    pd.DataFrame(list(elo.items()), columns=['team', 'elo'])
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
elo_table.index += 1
elo_table.head(25)

,team,elo
1,Spain,1942.284183
2,Argentina,1911.717219
3,France,1896.504051
4,England,1868.348663
5,Morocco,1864.229091
6,Portugal,1858.071224
7,Brazil,1850.648209
8,Japan,1839.961950
9,Colombia,1836.337063
10,Germany,1832.234995


## 6 · Form der letzten 5 Spiele

In [6]:
FORM_WEIGHTS = np.array([0.40, 0.25, 0.17, 0.11, 0.07])
FORM_N = len(FORM_WEIGHTS)

def match_points(sf, sa):
    if sf > sa:  return 1.0
    if sf == sa: return 0.5
    return 0.0

def weighted_form(dq):
    pts = list(dq)
    if not pts: return 0.5
    w = FORM_WEIGHTS[:len(pts)].copy()
    w /= w.sum()
    return float(np.dot(pts, w))

recent = defaultdict(lambda: deque(maxlen=FORM_N))
form_home_list, form_away_list = [], []

for _, row in all_matches.iterrows():
    home, away = row['home_team'], row['away_team']
    hs, as_ = row['home_score'], row['away_score']
    form_home_list.append(weighted_form(recent[home]))
    form_away_list.append(weighted_form(recent[away]))
    recent[home].appendleft(match_points(hs, as_))
    recent[away].appendleft(match_points(as_, hs))

all_matches['form_home'] = form_home_list
all_matches['form_away'] = form_away_list
all_matches['form_diff'] = all_matches['form_home'] - all_matches['form_away']

def get_current_form(team):
    return weighted_form(recent[team])

print('Form-Feature berechnet.')

Form-Feature berechnet.


## 7 · Poisson-Modell

In [7]:
model_matches = model_matches.join(
    all_matches[['elo_home','elo_away','elo_diff','form_home','form_away','form_diff']],
    how='left'
)

def tournament_weight(tournament):
    t = str(tournament)
    if 'FIFA World Cup' in t:           return 20
    if 'UEFA Euro' in t:                return 15
    if 'Copa America' in t:             return 15
    if 'African Cup of Nations' in t:   return 15
    if 'AFC Asian Cup' in t:            return 15
    if 'OFC Nations Cup' in t:          return 15
    if 'Nations League' in t:           return 8
    if 'qualification' in t.lower():    return 8
    if 'Friendly' in t:                 return 1
    return 2

model_matches['tournament_weight'] = model_matches['tournament'].apply(tournament_weight)
days_old = (pd.Timestamp.now() - model_matches['date']).dt.days
model_matches['time_weight'] = np.exp(-days_old / 1460)
model_matches['weight'] = model_matches['tournament_weight'] * model_matches['time_weight']

home_df = model_matches[['home_team','away_team','home_score','weight','elo_diff','form_diff']].copy()
home_df.columns = ['team','opponent','goals','weight','elo_diff','form_diff']

away_df = model_matches[['away_team','home_team','away_score','weight','elo_diff','form_diff']].copy()
away_df.columns = ['team','opponent','goals','weight','elo_diff','form_diff']
away_df['elo_diff']  = -away_df['elo_diff']
away_df['form_diff'] = -away_df['form_diff']

goal_model_data = pd.concat([home_df, away_df], ignore_index=True)

poisson_model = smf.glm(
    formula='goals ~ team + opponent + elo_diff + form_diff',
    data=goal_model_data,
    family=sm.families.Poisson(),
    freq_weights=goal_model_data['weight']
).fit()

print('Poisson-Modell trainiert.')
print(f'AIC: {poisson_model.aic:.1f}')

Poisson-Modell trainiert.
AIC: 205593.4


## 8 · Dixon-Coles-Korrektur

In [8]:
RHO = 0.07

def dixon_coles_tau(x, y, mu1, mu2, rho=RHO):
    if   x == 0 and y == 0: return 1.0 - mu1 * mu2 * rho
    elif x == 1 and y == 0: return 1.0 + mu2 * rho
    elif x == 0 and y == 1: return 1.0 + mu1 * rho
    elif x == 1 and y == 1: return 1.0 - rho
    return 1.0

print(f'Dixon-Coles RHO = {RHO}')

Dixon-Coles RHO = 0.07


## 9 · Vorhersagefunktionen mit Quoten-Ensemble

In [9]:
ODDS_TO_MODEL = {
    'USA':                  'United States',
    'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
    'Ivory Coast':          'Ivory Coast',
    'DR Congo':             'DR Congo',
    'Czech Republic':       'Czech Republic',
    'South Korea':          'South Korea',
    'New Zealand':          'New Zealand',
    'Saudi Arabia':         'Saudi Arabia',
    'Cape Verde':           'Cape Verde',
    'South Africa':         'South Africa',
}

GROUP_TO_MODEL = {
    'Czech Republic':         'Czech Republic',
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
    'United States':          'United States',
    'Ivory Coast':            'Ivory Coast',
    'New Zealand':            'New Zealand',
    'Saudi Arabia':           'Saudi Arabia',
    'Cape Verde':             'Cape Verde',
    'South Africa':           'South Africa',
    'South Korea':            'South Korea',
    'DR Congo':               'DR Congo',
    'Curacao':                'Curacao',
}

def model_name(team):
    return GROUP_TO_MODEL.get(team, team)

def odds_name(team):
    rev = {v: k for k, v in ODDS_TO_MODEL.items()}
    return rev.get(team, team)

def get_odds_probs(team1, team2):
    t1o, t2o = odds_name(team1), odds_name(team2)
    if (t1o, t2o) in odds_lookup:
        p = odds_lookup[(t1o, t2o)]
        return p['home_win'], p['draw'], p['away_win']
    if (t2o, t1o) in odds_lookup:
        p = odds_lookup[(t2o, t1o)]
        return p['away_win'], p['draw'], p['home_win']
    return None

def predict_goals(team, opponent):
    params    = poisson_model.params
    elo_diff  = get_elo(team) - get_elo(opponent)
    form_diff = get_current_form(team) - get_current_form(opponent)
    lp  = float(params.get('Intercept', 0.0))
    lp += float(params.get(f'team[T.{team}]', 0.0))
    lp += float(params.get(f'opponent[T.{opponent}]', 0.0))
    lp += float(params.get('elo_diff', 0.0)) * elo_diff
    lp += float(params.get('form_diff', 0.0)) * form_diff
    return float(np.exp(lp))

def dc_matrix(team1, team2, max_goals=8):
    g1 = predict_goals(model_name(team1), model_name(team2))
    g2 = predict_goals(model_name(team2), model_name(team1))
    matrix = np.outer(
        poisson.pmf(range(max_goals + 1), g1),
        poisson.pmf(range(max_goals + 1), g2)
    )
    for x in range(2):
        for y in range(2):
            matrix[x, y] *= dixon_coles_tau(x, y, g1, g2)
    matrix /= matrix.sum()
    return matrix, g1, g2

def ensemble_probs(team1, team2):
    matrix, g1, g2 = dc_matrix(team1, team2)
    pm1 = float(np.sum(np.tril(matrix, -1)))
    pdm = float(np.sum(np.diag(matrix)))
    pm2 = float(np.sum(np.triu(matrix,  1)))
    odds_p = get_odds_probs(team1, team2)
    if odds_p is not None:
        po1, pdo, po2 = odds_p
        p1 = MODEL_WEIGHT * pm1 + ODDS_WEIGHT * po1
        pd_ = MODEL_WEIGHT * pdm + ODDS_WEIGHT * pdo
        p2 = MODEL_WEIGHT * pm2 + ODDS_WEIGHT * po2
        src = 'ensemble'
    else:
        p1, pd_, p2 = pm1, pdm, pm2
        src = 'model_only'
    total = p1 + pd_ + p2
    return p1/total, pd_/total, p2/total, g1, g2, src

def tipp_ergebnis(team1, team2, max_goals=8):
    p1, pd_, p2, g1, g2, src = ensemble_probs(team1, team2)
    matrix, _, _ = dc_matrix(team1, team2, max_goals)
    outcome = max([('win1', p1), ('draw', pd_), ('win2', p2)], key=lambda x: x[1])[0]
    best_score, best_prob = None, -1
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            p = matrix[i, j]
            if outcome == 'win1' and i > j and p > best_prob:
                best_score, best_prob = (i, j), p
            elif outcome == 'draw' and i == j and p > best_prob:
                best_score, best_prob = (i, j), p
            elif outcome == 'win2' and j > i and p > best_prob:
                best_score, best_prob = (i, j), p
    t1r, t2r = best_score
    return t1r, t2r, p1, pd_, p2, src

print('Vorhersagefunktionen bereit.')
t1r, t2r, pw1, pd_, pw2, src = tipp_ergebnis('Germany', 'France')
print(f'Sanity Germany vs France: {t1r}:{t2r} | {src}')
print(f'  Germany {pw1:.1%} | X {pd_:.1%} | France {pw2:.1%}')

Vorhersagefunktionen bereit.
Sanity Germany vs France: 0:1 | model_only
  Germany 35.0% | X 23.8% | France 41.2%


## 10 · Gruppen & Spielplan

In [10]:
GROUPS = {
    'A': ['Mexico',        'South Africa',           'South Korea',   'Czech Republic'],
    'B': ['Canada',        'Bosnia and Herzegovina', 'Qatar',         'Switzerland'],
    'C': ['Brazil',        'Morocco',                'Haiti',         'Scotland'],
    'D': ['United States', 'Paraguay',               'Australia',     'Turkey'],
    'E': ['Germany',       'Curacao',                'Ivory Coast',   'Ecuador'],
    'F': ['Netherlands',   'Japan',                  'Sweden',        'Tunisia'],
    'G': ['Belgium',       'Egypt',                  'Iran',          'New Zealand'],
    'H': ['Spain',         'Cape Verde',             'Saudi Arabia',  'Uruguay'],
    'I': ['France',        'Senegal',                'Iraq',          'Norway'],
    'J': ['Argentina',     'Algeria',                'Austria',       'Jordan'],
    'K': ['Portugal',      'DR Congo',               'Uzbekistan',    'Colombia'],
    'L': ['England',       'Croatia',                'Ghana',         'Panama'],
}

schedule = [
    (gname, teams[i], teams[j])
    for gname, teams in GROUPS.items()
    for i in range(len(teams))
    for j in range(i+1, len(teams))
]
print(f'{len(schedule)} Gruppenspiele (12 x 6)')

72 Gruppenspiele (12 x 6)


## 11 · Tipps berechnen

In [11]:
tipp_rows = []

for gname, t1, t2 in schedule:
    g1, g2, pw1, pd_, pw2, src = tipp_ergebnis(t1, t2)
    confidence = max(pw1, pd_, pw2)
    outcome_label = (
        f'{t1} Sieg' if pw1 >= pd_ and pw1 >= pw2
        else ('Unentschieden' if pd_ >= pw2 else f'{t2} Sieg')
    )
    t1o, t2o = odds_name(t1), odds_name(t2)
    datum_raw = odds_date_map.get((t1o, t2o)) or odds_date_map.get((t2o, t1o))
    datum = pd.to_datetime(datum_raw).strftime('%d.%m. %H:%M') if datum_raw else '-'

    tipp_rows.append({
        'Datum':    datum,
        '_sort':    datum_raw or '9999',
        'Gruppe':   gname,
        'Team 1':   t1,
        'Team 2':   t2,
        'Tipp':     f'{g1}:{g2}',
        'Ausgang':  outcome_label,
        'Conf':     confidence,
        'Conf_str': f'{confidence:.0%}',
        'T1 Sieg':  f'{pw1:.0%}',
        'X':        f'{pd_:.0%}',
        'T2 Sieg':  f'{pw2:.0%}',
        'Quelle':   src,
    })

tipp_df = pd.DataFrame(tipp_rows)
print(f'Tipps: {len(tipp_df)} Spiele')
print(f'Ensemble: {(tipp_df["Quelle"]=="ensemble").sum()} | Nur Modell: {(tipp_df["Quelle"]=="model_only").sum()}')

Tipps: 72 Spiele
Ensemble: 69 | Nur Modell: 3


## 12 · Ausgabe nach Gruppen

In [12]:
for gname in sorted(GROUPS.keys()):
    grp = (
        tipp_df[tipp_df['Gruppe'] == gname]
        [['Team 1','Team 2','Tipp','Ausgang','Conf_str','T1 Sieg','X','T2 Sieg','Quelle']]
        .rename(columns={'Conf_str': 'Confidence'})
        .reset_index(drop=True)
    )
    grp.index += 1
    print(f'\n{"="*65}')
    print(f'  GRUPPE {gname}:  {", ".join(GROUPS[gname])}')
    print(f'{"="*65}')
    display(grp)


  GRUPPE A:  Mexico, South Africa, South Korea, Czech Republic


,Team 1,Team 2,Tipp,Ausgang,Confidence,T1 Sieg,X,T2 Sieg,Quelle
1,Mexico,South Africa,1:0,Mexico Sieg,59%,59%,24%,17%,ensemble
2,Mexico,South Korea,1:0,Mexico Sieg,45%,45%,28%,27%,ensemble
3,Mexico,Czech Republic,1:0,Mexico Sieg,46%,46%,27%,27%,ensemble
4,South Africa,South Korea,0:1,South Korea Sieg,49%,24%,27%,49%,ensemble
5,South Africa,Czech Republic,0:1,Czech Republic Sieg,49%,25%,27%,49%,ensemble
6,South Korea,Czech Republic,1:0,South Korea Sieg,37%,37%,29%,35%,ensemble



  GRUPPE B:  Canada, Bosnia and Herzegovina, Qatar, Switzerland


,Team 1,Team 2,Tipp,Ausgang,Confidence,T1 Sieg,X,T2 Sieg,Quelle
1,Canada,Bosnia and Herzegovina,1:0,Canada Sieg,51%,51%,25%,24%,ensemble
2,Canada,Qatar,1:0,Canada Sieg,66%,66%,20%,14%,ensemble
3,Canada,Switzerland,0:1,Switzerland Sieg,50%,24%,26%,50%,ensemble
4,Bosnia and Herzegovina,Qatar,2:1,Bosnia and Herzegovina Sieg,55%,55%,23%,21%,ensemble
5,Bosnia and Herzegovina,Switzerland,0:2,Switzerland Sieg,63%,16%,21%,63%,ensemble
6,Qatar,Switzerland,0:3,Switzerland Sieg,79%,8%,13%,79%,ensemble



  GRUPPE C:  Brazil, Morocco, Haiti, Scotland


...

,Team 1,Team 2,Tipp,Ausgang,Confidence,T1 Sieg,X,T2 Sieg,Quelle
1,Argentina,Algeria,2:0,Argentina Sieg,70%,70%,20%,11%,ensemble
2,Argentina,Austria,1:0,Argentina Sieg,59%,59%,24%,17%,ensemble
3,Argentina,Jordan,2:0,Argentina Sieg,79%,79%,14%,7%,ensemble
4,Algeria,Austria,0:1,Austria Sieg,43%,29%,28%,43%,ensemble
5,Algeria,Jordan,1:0,Algeria Sieg,56%,56%,24%,21%,ensemble
6,Austria,Jordan,1:0,Austria Sieg,66%,66%,19%,14%,ensemble



  GRUPPE K:  Portugal, DR Congo, Uzbekistan, Colombia


,Team 1,Team 2,Tipp,Ausgang,Confidence,T1 Sieg,X,T2 Sieg,Quelle
1,Portugal,DR Congo,1:0,Portugal Sieg,73%,73%,17%,9%,ensemble
2,Portugal,Uzbekistan,1:0,Portugal Sieg,74%,74%,17%,10%,ensemble
3,Portugal,Colombia,1:0,Portugal Sieg,46%,46%,26%,28%,ensemble
4,DR Congo,Uzbekistan,1:0,DR Congo Sieg,38%,38%,30%,32%,ensemble
5,DR Congo,Colombia,0:1,Colombia Sieg,63%,14%,22%,63%,ensemble
6,Uzbekistan,Colombia,0:1,Colombia Sieg,64%,14%,22%,64%,ensemble



  GRUPPE L:  England, Croatia, Ghana, Panama


,Team 1,Team 2,Tipp,Ausgang,Confidence,T1 Sieg,X,T2 Sieg,Quelle
1,England,Croatia,1:0,England Sieg,55%,55%,25%,20%,ensemble
2,England,Ghana,2:0,England Sieg,76%,76%,16%,8%,ensemble
3,England,Panama,2:0,England Sieg,78%,78%,13%,9%,ensemble
4,Croatia,Ghana,1:0,Croatia Sieg,62%,62%,23%,15%,ensemble
5,Croatia,Panama,2:0,Croatia Sieg,68%,68%,21%,12%,ensemble
6,Ghana,Panama,1:0,Ghana Sieg,46%,46%,27%,27%,ensemble


## 13 · Detail-Ansicht einzelner Spiele

In [13]:
def match_detail(team1, team2):
    g1, g2, pw1, pd_, pw2, src = tipp_ergebnis(team1, team2)
    matrix, mu1, mu2 = dc_matrix(team1, team2)
    rows = [(i, j, matrix[i,j]) for i in range(9) for j in range(9)]
    rows.sort(key=lambda x: x[2], reverse=True)
    top10 = pd.DataFrame(rows[:10], columns=[team1, team2, 'Wahrscheinlichkeit'])
    top10[[team1, team2]] = top10[[team1, team2]].astype(int)
    top10['Wahrscheinlichkeit'] = top10['Wahrscheinlichkeit'].map('{:.2%}'.format)
    odds_p = get_odds_probs(team1, team2)
    print(f'\n{"="*50}')
    print(f'  {team1}  vs  {team2}')
    print(f'{"="*50}')
    print(f'  Elo:       {get_elo(model_name(team1)):.0f} – {get_elo(model_name(team2)):.0f}')
    print(f'  Erw.Tore:  {mu1:.2f} – {mu2:.2f}  (Poisson)')
    print(f'  Quelle:    {src}')
    print(f'  {team1:<20} Sieg:  {pw1:.1%}')
    print(f'  {"Unentschieden":<20}        {pd_:.1%}')
    print(f'  {team2:<20} Sieg:  {pw2:.1%}')
    if odds_p:
        print(f'  Quoten: {team1} {odds_p[0]:.1%} | X {odds_p[1]:.1%} | {team2} {odds_p[2]:.1%}')
    print(f'  TIPP: {team1} {g1}:{g2} {team2}')
    display(top10)

match_detail('Germany', 'Ivory Coast')
match_detail('France', 'Norway')
match_detail('Brazil', 'Morocco')


  Germany  vs  Ivory Coast
  Elo:       1832 – 1732
  Erw.Tore:  1.95 – 0.74  (Poisson)
  Quelle:    ensemble
  Germany              Sieg:  62.8%
  Unentschieden               20.9%
  Ivory Coast          Sieg:  16.3%
  Quoten: Germany 60.2% | X 22.0% | Ivory Coast 17.9%
  TIPP: Germany 1:0 Ivory Coast


,Germany,Ivory Coast,Wahrscheinlichkeit
0,1,0,13.90%
1,2,0,12.91%
2,2,1,9.55%
3,1,1,9.09%
4,3,0,8.41%
5,3,1,6.22%
6,0,0,6.08%
7,0,1,5.69%
8,4,0,4.11%
9,1,2,3.62%



  France  vs  Norway
  Elo:       1897 – 1801
  Erw.Tore:  1.73 – 1.09  (Poisson)
  Quelle:    ensemble
  France               Sieg:  52.1%
  Unentschieden               24.7%
  Norway               Sieg:  23.2%
  Quoten: France 51.4% | X 26.3% | Norway 22.3%
  TIPP: France 1:0 Norway


,France,Norway,Wahrscheinlichkeit
0,1,0,11.15%
1,1,1,10.46%
2,2,1,9.73%
3,2,0,8.96%
4,0,1,7.29%
5,1,2,6.11%
6,3,1,5.61%
7,2,2,5.28%
8,0,0,5.20%
9,3,0,5.16%



  Brazil  vs  Morocco
  Elo:       1851 – 1864
  Erw.Tore:  0.96 – 0.87  (Poisson)
  Quelle:    ensemble
  Brazil               Sieg:  50.1%
  Unentschieden               27.3%
  Morocco              Sieg:  22.6%
  Quoten: Brazil 58.7% | X 25.1% | Morocco 16.2%
  TIPP: Brazil 1:0 Morocco


,Brazil,Morocco,Wahrscheinlichkeit
0,1,0,16.38%
1,0,0,15.19%
2,0,1,14.93%
3,1,1,12.45%
4,2,0,7.39%
5,2,1,6.41%
6,0,2,6.07%
7,1,2,5.81%
8,2,2,2.78%
9,3,0,2.36%


## 14 · Finaler Tippschein – sortiert nach Anpfiff

In [16]:
%pip install jinja2

final_tipp = (
    tipp_df
    .sort_values('_sort')
    .reset_index(drop=True)
)[['Datum','Gruppe','Team 1','Tipp','Team 2','Ausgang','Conf_str','Quelle']]
final_tipp = final_tipp.rename(columns={'Conf_str': 'Confidence'})
final_tipp.index += 1

def color_confidence(val):
    try:
        v = float(str(val).strip('%')) / 100
        if v >= 0.65:   return 'background-color: #27ae60; color: white'
        elif v >= 0.50: return 'background-color: #f39c12; color: white'
        else:           return 'background-color: #e74c3c; color: white'
    except:
        return ''

def color_source(val):
    if str(val) == 'ensemble': return 'background-color: #2980b9; color: white'
    return 'background-color: #7f8c8d; color: white'

styler = final_tipp.style.set_caption('WM 2026 – Kompletter Tippschein Gruppenphase')

# Kompatibel mit Pandas >= 2.1 (.map) und aelter (.applymap)
if hasattr(styler, 'map'):
    styler = styler.map(color_confidence, subset=['Confidence'])
    styler = styler.map(color_source,     subset=['Quelle'])
else:
    styler = styler.applymap(color_confidence, subset=['Confidence'])
    styler = styler.applymap(color_source,     subset=['Quelle'])

display(styler)


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.


   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   ---------------------------------------- 2/2 [jinja2]



,Datum,Gruppe,Team 1,Tipp,Team 2,Ausgang,Confidence,Quelle
1,11.06. 19:00,A,Mexico,1:0,South Africa,Mexico Sieg,59%,ensemble
2,12.06. 02:00,A,South Korea,1:0,Czech Republic,South Korea Sieg,37%,ensemble
3,12.06. 19:00,B,Canada,1:0,Bosnia and Herzegovina,Canada Sieg,51%,ensemble
4,13.06. 01:00,D,United States,1:0,Paraguay,United States Sieg,41%,ensemble
5,13.06. 19:00,B,Qatar,0:3,Switzerland,Switzerland Sieg,79%,ensemble
6,13.06. 22:00,C,Brazil,1:0,Morocco,Brazil Sieg,50%,ensemble
7,14.06. 01:00,C,Haiti,0:2,Scotland,Scotland Sieg,72%,ensemble
8,14.06. 04:00,D,Australia,0:1,Turkey,Turkey Sieg,49%,ensemble
9,14.06. 20:00,F,Netherlands,1:0,Japan,Netherlands Sieg,47%,ensemble
10,14.06. 23:00,E,Ivory Coast,0:1,Ecuador,Ecuador Sieg,43%,ensemble


## 15 · Heatmap – Confidence je Gruppe

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for ax_idx, (gname, teams) in enumerate(sorted(GROUPS.items())):
    ax = axes[ax_idx]
    n = len(teams)
    conf_matrix = np.full((n, n), np.nan)
    labels = [[''] * n for _ in range(n)]

    for i, ta in enumerate(teams):
        for j, tb in enumerate(teams):
            if i == j: continue
            if i < j:
                g1, g2, pw1, pd_, pw2, _ = tipp_ergebnis(ta, tb)
                conf = max(pw1, pd_, pw2)
                conf_matrix[i,j] = conf
                labels[i][j] = f'{g1}:{g2}\n{conf:.0%}'
            else:
                g2r, g1r, pw2r, pd_r, pw1r, _ = tipp_ergebnis(teams[j], teams[i])
                conf = max(pw1r, pd_r, pw2r)
                conf_matrix[i,j] = conf
                labels[i][j] = f'{g1r}:{g2r}\n{conf:.0%}'

    masked = np.ma.masked_invalid(conf_matrix)
    ax.imshow(masked, cmap=plt.cm.RdYlGn, vmin=0.3, vmax=0.8, aspect='auto')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    short = [t[:10] for t in teams]
    ax.set_xticklabels(short, rotation=30, ha='right', fontsize=8)
    ax.set_yticklabels(short, fontsize=8)
    for i in range(n):
        for j in range(n):
            if labels[i][j]:
                ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=7, fontweight='bold')
    ax.set_title(f'Gruppe {gname}', fontsize=11, fontweight='bold')

plt.suptitle('WM 2026 – Tipp-Heatmap Gruppenphase', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 16 · Monte-Carlo – Gruppenplatz-Wahrscheinlichkeiten

In [ ]:
def simulate_match_ensemble(team1, team2):
    p1, pd_, p2, _, _, _ = ensemble_probs(team1, team2)
    matrix, _, _ = dc_matrix(team1, team2)
    outcome = np.random.choice(['win1','draw','win2'], p=[p1, pd_, p2])
    n = matrix.shape[0]
    mask = np.zeros_like(matrix)
    for i in range(n):
        for j in range(n):
            if   outcome == 'win1' and i > j:  mask[i,j] = matrix[i,j]
            elif outcome == 'draw' and i == j: mask[i,j] = matrix[i,j]
            elif outcome == 'win2' and j > i:  mask[i,j] = matrix[i,j]
    flat = mask.flatten()
    s = flat.sum()
    if s == 0:
        return (1,0) if outcome=='win1' else ((0,0) if outcome=='draw' else (0,1))
    idx = np.random.choice(len(flat), p=flat/s)
    r, c = divmod(idx, n)
    return int(r), int(c)

def simulate_group_ensemble(teams):
    table = {t: [0,0,0,0] for t in teams}
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            t1, t2 = teams[i], teams[j]
            g1, g2 = simulate_match_ensemble(t1, t2)
            table[t1][1]+=g1; table[t1][2]+=g2; table[t1][3]+=g1-g2
            table[t2][1]+=g2; table[t2][2]+=g1; table[t2][3]+=g2-g1
            if g1>g2:   table[t1][0]+=3
            elif g1<g2: table[t2][0]+=3
            else:       table[t1][0]+=1; table[t2][0]+=1
    return sorted(table.items(), key=lambda x:(x[1][0],x[1][3],x[1][1]), reverse=True)

all_teams_flat = [t for g in GROUPS.values() for t in g]
reach = {t:{s:0 for s in ['1st','2nd','3rd','4th']} for t in all_teams_flat}

N_SIM = 10_000
print(f'Starte {N_SIM:,} Simulationen ...')
t0 = time.time()

for _ in range(N_SIM):
    for gname, teams in GROUPS.items():
        for pos,(team,_) in enumerate(simulate_group_ensemble(teams)):
            reach[team][['1st','2nd','3rd','4th'][pos]] += 1

print(f'Fertig in {time.time()-t0:.1f}s')

In [ ]:
sim_rows = []
for gname, teams in sorted(GROUPS.items()):
    for team in teams:
        r = reach[team]
        sim_rows.append({
            'Gruppe':     gname,
            'Team':       team,
            'Elo':        round(get_elo(model_name(team))),
            '1. Platz %': round(r['1st']/N_SIM*100, 1),
            '2. Platz %': round(r['2nd']/N_SIM*100, 1),
            '3. Platz %': round(r['3rd']/N_SIM*100, 1),
            '4. Platz %': round(r['4th']/N_SIM*100, 1),
        })

sim_df = pd.DataFrame(sim_rows)

for gname in sorted(GROUPS.keys()):
    grp = (
        sim_df[sim_df['Gruppe']==gname]
        .sort_values('1. Platz %', ascending=False)
        .reset_index(drop=True)
    )
    grp.index += 1
    print(f'\n{"="*50}  GRUPPE {gname}  {"="*5}')
    display(grp.drop(columns=['Gruppe']))

## 17 · Sensitivitätsanalyse – Quoten-Gewichtung

In [ ]:
weights_test = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
test_matches = [
    ('Germany',   'Ivory Coast'),
    ('France',    'Norway'),
    ('Brazil',    'Morocco'),
    ('Argentina', 'Algeria'),
    ('Spain',     'Uruguay'),
]

print(f'{"Spiel":<32} | ', end='')
for w in weights_test:
    print(f'  w={w:.1f}  ', end='')
print()
print('-' * 100)

for t1, t2 in test_matches:
    print(f'{t1:<15} vs {t2:<15} | ', end='')
    matrix, _, _ = dc_matrix(t1, t2)
    pm1 = float(np.sum(np.tril(matrix,-1)))
    pdm = float(np.sum(np.diag(matrix)))
    pm2 = float(np.sum(np.triu(matrix, 1)))
    odds_p = get_odds_probs(t1, t2)
    for w in weights_test:
        if odds_p:
            pw1 = (1-w)*pm1 + w*odds_p[0]
            pdr = (1-w)*pdm + w*odds_p[1]
            pw2 = (1-w)*pm2 + w*odds_p[2]
        else:
            pw1, pdr, pw2 = pm1, pdm, pm2
        outcome = 'S1' if pw1>=pdr and pw1>=pw2 else ('X' if pdr>=pw2 else 'S2')
        print(f'  {outcome}({max(pw1,pdr,pw2):.0%}) ', end='')
    print()